In [0]:
# Get last processed timestamp

last_ts = spark.sql("""
SELECT last_processed_timestamp 
FROM dev_catalog.pipeline_schema.pipeline_metadata 
WHERE pipeline_name = 'api_pipeline'
""").collect()[0][0]

print(last_ts)

In [0]:
##Fetching data from bronze table

from pyspark.sql.functions import col

bronze_df = spark.read.format("delta") \
    .load("/Volumes/dev_catalog/pipeline_schema/bronze_table/api_data")

incremental_df = bronze_df.filter(col("ingestion_time") > last_ts)

In [0]:
#Data Quality framework

#Step 1: DQ Functions

from pyspark.sql.functions import col

def check_nulls(df, columns):
    return {c: df.filter(col(c).isNull()).count() for c in columns}

def check_duplicates(df, column):
    return df.groupBy(column).count().filter("count > 1").count()

def check_invalid_numeric(df, column):
    return df.filter(~col(column).rlike("^[0-9.]+$")).count()

def check_invalid_timestamp(df, column):
    return df.filter(~col(column).rlike("^[0-9T:\\-\\.Z]+$")).count()

dq_bronze = {}

dq_bronze["nulls"] = check_nulls(incremental_df, ["id", "amount", "updated_at","name","status"])
dq_bronze["duplicate_ids"] = check_duplicates(incremental_df, "id")
dq_bronze["invalid_amount"] = check_invalid_numeric(incremental_df, "amount")
dq_bronze["invalid_timestamp"] = check_invalid_timestamp(incremental_df, "updated_at")

print("Bronze DQ:", dq_bronze)

In [0]:
#Casting and cleaning

from pyspark.sql.functions import when, to_timestamp, col

transformed_df = incremental_df

# Fix amount
transformed_df = transformed_df.withColumn(
    "amount_clean",
    when(col("amount").rlike("^[0-9.]+$"), col("amount").cast("double"))
)

# Fix timestamp
transformed_df = transformed_df.withColumn(
    "updated_at_clean",
    to_timestamp(col("updated_at"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
)

# Fix null names
transformed_df = transformed_df.withColumn(
    "name_clean",
    when(col("name").isNull(), "Unknown").otherwise(col("name"))
)

# Fix null status
transformed_df = transformed_df.withColumn(
    "status_clean",
    when(col("status").isNull(), "Pending").otherwise(col("status"))
)

In [0]:
# Spliting  good and bad records

from pyspark.sql.functions import col, when

# GOOD RECORDS → Only clean, final columns
good_df = transformed_df.filter(
    col("id").isNotNull() &
    col("amount_clean").isNotNull() &
    col("updated_at_clean").isNotNull()
).select(
    col("id"),
    col("name_clean").alias("name"),
    col("status_clean").alias("status"),
    col("amount_clean").alias("amount"),
    col("updated_at_clean").alias("updated_at"),
    col("source"),
    col("ingestion_time")
)

# BAD RECORDS → Keep raw + add rejection reason
from pyspark.sql.functions import concat_ws

bad_df = transformed_df.withColumn(
    "rejection_reason",
    concat_ws(", ",
        when(col("id").isNull(), "Missing ID"),
        when(col("amount_clean").isNull(), "Invalid amount"),
        when(col("updated_at_clean").isNull(), "Invalid timestamp")
    )
).filter(col("rejection_reason") != "")

In [0]:
#Again DQ check before inserting cleaned data into silver table

from pyspark.sql.functions import col

def check_nulls(df, columns):
    return {c: df.filter(col(c).isNull()).count() for c in columns}

def check_duplicates(df, column):
    return df.groupBy(column).count().filter("count > 1").count()

def check_invalid_numeric(df, column):
    return df.filter(~col(column).rlike("^[0-9.]+$")).count()

def check_invalid_timestamp(df, column):
    return df.filter(~col(column).rlike("^[0-9T:\\-\\.Z]+$")).count()

dq_bronze = {}

dq_bronze["nulls"] = check_nulls(good_df, ["id", "amount", "updated_at","name","status"])
dq_bronze["duplicate_ids"] = check_duplicates(good_df, "id")
dq_bronze["invalid_amount"] = check_invalid_numeric(good_df, "amount")
dq_bronze["invalid_timestamp"] = check_invalid_timestamp(good_df, "updated_at")

print("Bronze DQ:", dq_bronze)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dev_catalog.pipeline_schema.silver_table (
    id LONG,
    name STRING,
    status STRING,
    amount DOUBLE,
    updated_at TIMESTAMP,
    source STRING,
    ingestion_time TIMESTAMP
)
USING DELTA
""")

In [0]:
#Using Merge to update table with new incremental data

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "dev_catalog.pipeline_schema.silver_table"
)

delta_table.alias("target").merge(
    good_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
bad_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("dev_catalog.pipeline_schema.bad_records")
#("/Volumes/dev_catalog/pipeline_schema/quarantine/api_bad")

In [0]:
#Update watermark so that it can be used in incremental load to cross check last processed date

from pyspark.sql.functions import max

new_ts = incremental_df.select(max("ingestion_time")).collect()[0][0]

spark.sql(f"""
UPDATE dev_catalog.pipeline_schema.pipeline_metadata
SET last_processed_timestamp = TIMESTAMP('{new_ts}')
WHERE pipeline_name = 'api_pipeline'
""")


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dev_catalog.pipeline_schema.pipeline_runs (
    run_id STRING,
    run_time TIMESTAMP,
    status STRING,
    records_processed INT,
    bad_records INT
)
""")

In [0]:
#The error:

# Failed to merge fields 'records_processed' and 'records_processed'. SQLSTATE: 22005

# means that the data type Spark infers for records_processed in your run_log DataFrame does not match the existing records_processed column in the Delta table dev_catalog.pipeline_schema.pipeline_runs (even though both look like integers).

# Most likely:

# The table expects INTEGER but Spark inferred BIGINT (or vice versa) from the count, or

# The column already exists in the table with a different numeric type and Spark is trying to “merge” schemas on append and fails.

# Quick fix
# Force the types explicitly in your DataFrame:


from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import uuid

run_id = str(uuid.uuid4())

good_count = good_df.count()
bad_count = bad_df.count()

# Explicit schema to match the table
schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("status", StringType(), False),
    StructField("records_processed", IntegerType(), False),
    StructField("bad_records", IntegerType(), False),
])

run_log = spark.createDataFrame(
    [(run_id, "SUCCESS", good_count, bad_count)],
    schema=schema
)

run_log = run_log.withColumn("run_time", current_timestamp())

run_log = run_log.select(
    "run_id",
    "run_time",
    "status",
    "records_processed",
    "bad_records"
)

run_log.write \
    .mode("append") \
    .saveAsTable("dev_catalog.pipeline_schema.pipeline_runs")

In [0]:
# DQ Metrices table

from pyspark.sql import Row

total = bronze_df.count()
good = good_df.count()
bad = bad_df.count()

dq_metrics = {
    "total_records": total,
    "good_records": good,
    "bad_records": bad,
    "success_rate": (good / total) * 100
}

dq_rows = [Row(metric=k, value=str(v)) for k, v in dq_metrics.items()]
dq_df = spark.createDataFrame(dq_rows)

dq_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("dev_catalog.pipeline_schema.dq_metrics")

In [0]:
%sql
select count(*) from dev_catalog.pipeline_schema.silver_table;